# Step 10 — **SAM3D from gsplat-rendered views** (AffordSplat / 3DGS → mesh + latents)

**Pipeline position:** after **`03_rendering_gaussian_splat.ipynb`** (or any path to a 3DGS `.ply`), before VLM/projection if you want mesh-centric features.

**What this does:**
1. **True gsplat** rasterisation → `run_dir/sam3d_dataset/images/view_XXX.png` + full-foreground masks (SAM3D layout).
2. **SAM3D** on **`view_{REFERENCE_VIEW:03d}.png`** (single-image API; extra views are for debugging / future multi-view fusion).
3. Writes **`run_dir/reconstruction/`** (`mesh.glb`, `gaussian.ply`, `shape_latent.pt`, …) and optionally **`data/cache/sam3d/<stem>/global_latent.pt`** per `configs/default.yaml`.

**Requirements:** CUDA + **`gsplat`** + **SAM3D weights** (`sam-3d-objects/…`, `reconstruction.sam3d_config`). The last cell catches failures and prints the error.

**You cannot use this outside the container** in a supported way: the gsplat → SAM3D stack is pinned and tested only in the **`sam3d-pipeline` Docker** image (see [docker/README.md](../docker/README.md)). Run this notebook or the CLI **inside** `docker compose run --rm sam3d-pipeline bash` with the repo mounted at `/workspace`.

**CLI (from `/workspace` in the container):** `PYTHONPATH=src python scripts/render_gsplat_and_sam3d.py --splat_path … --run_dir …`

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for ROOT in [_cwd, *_cwd.parents]:
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
        if str(ROOT / "src") not in sys.path:
            sys.path.insert(0, str(ROOT / "src"))
        break
else:
    raise RuntimeError("Run from the repo or notebooks/.")

from utils.config import load_config

_cfg = load_config()
print("ROOT:", ROOT)


In [ ]:
# --- Edit: input splat and run directory (under exports/ so Cursor shows outputs) ---
from pathlib import Path

import torch

from datasets.affordsplat_local_dataset import resolve_affordsplat_root, sample_random_affordsplat_row

RUN_STEM = "demo_gsplat_sam3d"
RUN_DIR = ROOT / "exports" / "gsplat_sam3d_runs" / RUN_STEM
REFERENCE_VIEW = 0

# Set explicitly, or leave None to sample a random AffordSplat training Gaussian when mirror exists.
SPLAT_PLY: Path | None = None

_spl = SPLAT_PLY
if _spl is None:
    root = resolve_affordsplat_root(_cfg)
    if root is not None:
        row = sample_random_affordsplat_row(cfg=_cfg, subset="Seen", split="train", seed=None)
        if row is not None:
            _spl = row.splat_path
            print("Auto splat:", _spl, "|", row.sample_id)
if _spl is None or not Path(_spl).is_file():
    raise FileNotFoundError(
        "Set SPLAT_PLY to a .ply, or install AffordSplat under AFFORDANCE_DATA_ROOT / /workspace/data."
    )

SPLAT_PLY = Path(_spl).resolve()
print("CUDA:", torch.cuda.is_available(), "| SPLAT_PLY:", SPLAT_PLY)
print("RUN_DIR:", RUN_DIR.resolve())


In [ ]:
from reconstruction.gsplat_to_sam3d import gsplat_ply_to_sam3d_reconstruction

RUN_DIR.mkdir(parents=True, exist_ok=True)

try:
    out = gsplat_ply_to_sam3d_reconstruction(
        SPLAT_PLY,
        RUN_DIR,
        cfg=_cfg,
        object_stem=None,
        reference_view_index=REFERENCE_VIEW,
        gsplat_seed=None,
        sam3d_seed=42,
        cache_global_latent=None,
    )
except Exception as exc:
    print("gsplat / SAM3D failed:", type(exc).__name__, exc)
    hint = (
        "Check: CUDA + gsplat for raster; SAM3D checkout with notebook/inference.py "
        "(git submodule update --init sam-3d-objects or SAM3D_OBJECTS_ROOT); "
        "HF checkpoints under reconstruction.sam3d_config (sam-3d-objects/doc/setup.md §2)."
    )
    if isinstance(exc, ModuleNotFoundError) and getattr(exc, "name", None) == "inference":
        hint = (
            "SAM3D code path: clone/init sam-3d-objects (notebook/inference.py) or set SAM3D_OBJECTS_ROOT; "
            "Docker bind mount must not hide /workspace/sam-3d-objects with an empty folder. "
            "Then HF checkpoints (reconstruction.sam3d_config)."
        )
    elif isinstance(exc, FileNotFoundError) and "SAM3D pipeline config" in str(exc):
        hint = ""  # Exception text already has HF download commands
    elif isinstance(exc, FileNotFoundError) and "pipeline.yaml" in str(exc):
        hint = (
            "Missing SAM3D weights: download facebook/sam-3d-objects on HF into "
            "sam-3d-objects/checkpoints/hf/ (see sam-3d-objects/doc/setup.md §2; hf auth login / HF_TOKEN)."
        )
    if hint:
        print(hint)
else:
    print("dataset_dir:", out["dataset_dir"])
    print("reconstruction_dir:", out["reconstruction_dir"])
    print("mesh:", out["paths"]["mesh"])
    if "global_latent_path" in out:
        print("global_latent:", out["global_latent_path"])
    print()
    print("Downstream: in notebooks 02, 04, 05, 06 set")
    print("  SAM3D_RUN_DIR =", repr(str(RUN_DIR.resolve())))
    print("then clear outputs/notebooks/02_rendering/ if you switch meshes, and rerun 02→04→05→06.")


In [ ]:
# Optional: show the reference RGB written for SAM3D
from IPython.display import Image, display

ref = RUN_DIR / "sam3d_dataset" / "images" / f"view_{REFERENCE_VIEW:03d}.png"
if ref.is_file():
    display(Image(filename=str(ref)))
else:
    print("No preview (run previous cell successfully first).")
